In [1]:
import pandas as pd

df = pd.read_json('export.geojson')
df.head()

<ipython-input-1-5156b411b349>:3: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json('export.geojson')


<class 'ValueError'>: Expected object or value

In [3]:
import json
import pandas as pd

with open('export (2).geojson', encoding='utf-8') as f:
    data = json.load(f)

In [4]:
import json
import pandas as pd

with open('export (2).geojson', encoding='utf-8') as f:
    data = json.load(f)

In [5]:
features = data['features']

In [6]:
rows = []

for item in features:
    props = item['properties']
    coords = item['geometry']['coordinates']
    
    rows.append({
        'lat': coords[1],
        'lon': coords[0],
        'wheelchair': props.get('wheelchair', 0)
    })

df = pd.DataFrame(rows)

In [7]:
df['wheelchair'] = df['wheelchair'].apply(lambda x: 1 if x == 'yes' else 0)

In [8]:
df.to_csv('cleaned_data.csv', index=False)

df.head()

,lat,lon,wheelchair
0,55.778834,37.653721,1
1,55.723560,37.396826,1
2,55.726898,37.449586,0
3,55.715937,37.375260,0
4,55.814159,37.735580,0


In [9]:
def get_okrug(lat, lon):
    if lat > 55.8:
        return 'САО'
    elif lat > 55.75:
        if lon < 37.6:
            return 'СЗАО'
        else:
            return 'СВАО'
    elif lat > 55.7:
        return 'ЦАО'
    else:
        if lon < 37.6:
            return 'ЮЗАО'
        else:
            return 'ЮВАО'

df['okrug'] = df.apply(lambda row: get_okrug(row['lat'], row['lon']), axis=1)

In [10]:
okrug_stats = df.groupby('okrug')['wheelchair'].agg(['sum', 'count']).reset_index()

okrug_stats = okrug_stats.rename(columns={
    'sum': 'accessible_objects',
    'count': 'total_objects'
})

In [11]:
final = okrug_stats.merge(pop_df, on='okrug', how='left')

<class 'NameError'>: name 'pop_df' is not defined

In [12]:
pop_df = pd.read_csv('population.csv')
pop_df.head()

,okrug,population
0,ЦАО,↗774 430[8]
1,САО,↗1 217 909[8]
2,СВАО,↗1 455 811[8]
3,ВАО,↘1 508 678[8]
4,ЮВАО,↗1 515 787[8]


In [13]:
pop_df['population'] = pop_df['population'].str.replace(r'\D', '', regex=True).astype(int)

In [14]:
final = okrug_stats.merge(pop_df, on='okrug', how='left')


In [15]:
final['index'] = final['accessible_objects'] / final['population'] * 10000

In [16]:
final['share'] = final['accessible_objects'] / final['total_objects']


In [17]:
final.sort_values(by='index', ascending=False)

,okrug,accessible_objects,total_objects,population,index,share
3,ЦАО,2034,3911,7744308,2.626445,0.520072
4,ЮВАО,3564,4441,15157878,2.351253,0.802522
0,САО,1922,3759,12179098,1.578114,0.511306
1,СВАО,683,2208,14558118,0.469154,0.309330
5,ЮЗАО,537,1508,14355508,0.374072,0.356101
2,СЗАО,365,772,10395968,0.351098,0.472798
